# 00 — Theory: Building a Nano Vision-Language Model

## 1. What is a VLM?

A **Vision-Language Model (VLM)** is a model that learns to work with both **visual information** and **language**.

A traditional computer vision model receives an image:

```text
Image → Image Model → Visual Output
```

A language model receives text:

```text
Text → Language Model → Text Output
```

A **Vision-Language Model** brings these two modalities together:

```text
Image + Text → Shared Representation
```

This allows the model to learn relationships between visual concepts and language.

For example, consider this image:

> A blue circle at the top-left.

The model should learn that the visual content of the image corresponds to the meaning of the text:

> `blue circle top-left`

### Our NanoVLM

Our project will use two separate encoders:

```text
Image → Image Encoder → z_I
```

and

```text
Text → Text Encoder → z_T
```

where $z_I$ and $z_T$ are embeddings in the same vector space.

We then train the model so that matching image-text pairs have similar embeddings.

---

# 2. Image Encoder vs Text Encoder

The two modalities have fundamentally different forms.

An image is represented as a tensor:

$$
I \in \mathbb{R}^{H \times W \times C}
$$

For our project:

$$
I \in \mathbb{R}^{32 \times 32 \times 3}
$$

The image encoder converts this tensor into a compact vector:

$$
f_I(I) = z_I
$$

Our CNN will eventually produce:

$$
z_I \in \mathbb{R}^{64}
$$

Text, however, is represented using discrete tokens.

For example:

```text
"blue circle top-left"
```

might become:

```text
[CLS, blue, circle, top-left]
```

The text encoder converts these tokens into another vector:

$$
f_T(T) = z_T
$$

with:

$$
z_T \in \mathbb{R}^{64}
$$

Therefore:

```text
Image                         Text

32 × 32 × 3                   Tokens
     │                           │
     ▼                           ▼
Image Encoder                 Text Encoder
     │                           │
     ▼                           ▼
 64-d vector                 64-d vector
```

The crucial idea is that **both encoders produce vectors of the same dimensionality**.

---

# 3. Multimodal Fusion

There are several ways to combine visual and textual information.

### Early Fusion

The visual and textual representations are combined early and processed jointly.

```text
Image ─────┐
           ├──> Joint Model ──> Output
Text ──────┘
```

### Late Fusion

Each modality is encoded independently and the resulting representations are aligned.

```text
Image ──> Image Encoder ──> Image Embedding
                              │
                              │ Similarity
                              │
Text ───> Text Encoder ───> Text Embedding
```

This is the approach we will use.

It is similar in spirit to **CLIP-style contrastive learning**.

### Cross-Attention Fusion

A more sophisticated approach allows image tokens to attend to text tokens and vice versa.

This is common in larger VLM architectures.

For our NanoVLM, we deliberately avoid this complexity.

Our goal is to understand the fundamentals first.

---

# 4. Why Do Shared Embedding Spaces Work?

The central idea is to map different modalities into the **same mathematical space**.

Suppose we have:

$$
z_I = f_I(I)
$$

and

$$
z_T = f_T(T)
$$

Both vectors live in:

$$
\mathbb{R}^{64}
$$

This means we can directly compare them.

For example:

```text
                    Shared embedding space

              image: "blue circle"
                     ●
                    /
                   /
                  /
                 ●
              text: "blue circle"
```

A matching image and caption should be close together.

A mismatched caption should be farther away.

The model therefore learns:

$$
I_i \leftrightarrow T_i
$$

while separating:

$$
I_i \not\leftrightarrow T_j \quad \text{for } i \ne j
$$

This transforms multimodal understanding into a geometric learning problem.

Instead of explicitly telling the model:

> "This image means this sentence."

we train it to organize the embedding space so that matching concepts become close.

---

# 5. Positive and Negative Pairs

Suppose our dataset contains:

| Image | Caption |
| ----- | ------- |
| $I_1$ | $T_1$   |
| $I_2$ | $T_2$   |
| $I_3$ | $T_3$   |

The correct pairs are:

$$
(I_1,T_1), \quad (I_2,T_2), \quad (I_3,T_3)
$$

These are called **positive pairs**.

For example:

```text
Image: blue circle top-left

Caption: blue circle top-left
```

is a positive pair.

But:

```text
Image: blue circle top-left

Caption: red square bottom-right
```

is a **negative pair**.

With a batch of $N$ examples, we automatically obtain many negative pairs.

For example:

```text
        T1      T2      T3
I1     POS     NEG     NEG
I2     NEG     POS     NEG
I3     NEG     NEG     POS
```

This is one of the powerful ideas behind contrastive learning:

> **The other examples in the batch can act as negative examples.**

---

# 6. Similarity Matrix

After encoding a batch, suppose we have:

$$
Z_I =
\begin{bmatrix}
z_{I_1} \
z_{I_2} \
\vdots \
z_{I_N}
\end{bmatrix}
$$

and:

$$
Z_T =
\begin{bmatrix}
z_{T_1} \
z_{T_2} \
\vdots \
z_{T_N}
\end{bmatrix}
$$

We calculate similarities between every image and every text embedding.

This produces an $N \times N$ matrix:

$$
S_{ij} = \operatorname{sim}(z_{I_i}, z_{T_j})
$$

For example:

$$
S =
\begin{bmatrix}
0.95 & 0.15 & 0.05 \
0.20 & 0.90 & 0.30 \
0.10 & 0.25 & 0.92
\end{bmatrix}
$$

The diagonal contains the correct pairs:

$$
S_{11}, \quad S_{22}, \quad S_{33}
$$

Therefore, during training, we want the diagonal values to become large relative to the off-diagonal values.

Visually:

```text
                 Text
              T1    T2    T3

        ┌─────────────────────┐
  I1    │ ✓     ✗     ✗       │
        │                     │
  I2    │ ✗     ✓     ✗       │
        │                     │
  I3    │ ✗     ✗     ✓       │
        └─────────────────────┘
```

---

# 7. Embedding Normalization

Before calculating similarity, we normalize the embeddings.

For a vector $z$:

$$
\hat{z} = \frac{z}{|z|_2}
$$

where:

$$
|z|_2 = \sqrt{\sum_i z_i^2}
$$

After normalization:

$$
|\hat{z}|_2 = 1
$$

This means every embedding lies on the unit hypersphere.

We can then calculate similarity using the dot product:

$$
\operatorname{sim}(z_I,z_T) = \hat{z}_I^T \hat{z}_T
$$

For normalized vectors, this is equivalent to **cosine similarity**.

Therefore:

$$
-1 \leq \operatorname{sim}(z_I,z_T) \leq 1
$$

A value close to:

$$
1
$$

means the embeddings point in similar directions.

A value close to:

$$
0
$$

means they are approximately orthogonal.

A negative value indicates opposing directions.

### Why normalize?

Without normalization, the model could increase similarity simply by increasing the magnitude of the embeddings.

Normalization forces the model to learn meaningful **directions** rather than relying primarily on vector magnitude.

---

# 8. Temperature

The similarity matrix alone is not enough.

We introduce a temperature parameter:

$$
\tau
$$

and scale the similarities:

$$
\frac{S_{ij}}{\tau}
$$

The probability of selecting text $j$ for image $i$ becomes:

$$
p(j \mid i)
===========

\frac{\exp(S_{ij}/\tau)}
{\sum_{k=1}^{N}\exp(S_{ik}/\tau)}
$$

### What does temperature do?

Temperature controls how sharp the probability distribution becomes.

A **small $\tau$** makes the distribution sharper.

A **large $\tau$** makes it softer.

For example:

```text
Large temperature:

0.40   0.35   0.25


Small temperature:

0.95   0.04   0.01
```

The temperature therefore controls how strongly the model focuses on the most similar candidate.

In our implementation, temperature may be represented directly as a parameter or as a learnable `logit_scale`.

---

# 9. Contrastive Probability

For image $i$, the correct caption is $T_i$.

We calculate the probability that caption $j$ is the correct caption for image $i$:

$$
p(T_j \mid I_i)
===============

\frac{\exp(S_{ij}/\tau)}
{\sum_{k=1}^{N}\exp(S_{ik}/\tau)}
$$

The desired probability is therefore:

$$
p(T_i \mid I_i)
$$

We want:

$$
p(T_i \mid I_i) \rightarrow 1
$$

for every training example.

This converts similarity learning into a **classification problem**.

The model is effectively being asked:

> "Given this image, which caption in this batch is the correct one?"

---

# 10. Image-to-Text Loss

For each image, we want the corresponding text to receive the highest probability.

The loss for image $i$ is:

$$
L_i^{I\rightarrow T}
====================

-\log
\left(
\frac{\exp(S_{ii}/\tau)}
{\sum_{j=1}^{N}\exp(S_{ij}/\tau)}
\right)
$$

The complete image-to-text loss is:

$$
L_{I\rightarrow T}
==================

-\frac{1}{N}
\sum_{i=1}^{N}
\log
\left(
\frac{\exp(S_{ii}/\tau)}
{\sum_{j=1}^{N}\exp(S_{ij}/\tau)}
\right)
$$

In simple terms:

> For every image, classify its matching caption correctly among all captions in the batch.

---

# 11. Text-to-Image Loss

We can reverse the problem.

Instead of asking:

> "Which caption matches this image?"

we ask:

> "Which image matches this caption?"

The probability becomes:

$$
p(I_j \mid T_i)
===============

\frac{\exp(S_{ji}/\tau)}
{\sum_{k=1}^{N}\exp(S_{ki}/\tau)}
$$

The corresponding loss is:

$$
L_{T\rightarrow I}
==================

-\frac{1}{N}
\sum_{i=1}^{N}
\log
\left(
\frac{\exp(S_{ii}/\tau)}
{\sum_{j=1}^{N}\exp(S_{ji}/\tau)}
\right)
$$

Notice that we are effectively applying the same idea to the **columns** of the similarity matrix instead of the rows.

---

# 12. Symmetric CLIP Loss

We now combine the two directions.

**Image → Text:**

$$
L_{I\rightarrow T}
$$

**Text → Image:**

$$
L_{T\rightarrow I}
$$

The final contrastive loss is:

$$
L =
\frac{1}{2}
\left(
L_{I\rightarrow T}
+
L_{T\rightarrow I}
\right)
$$

This is the symmetric CLIP-style objective that our NanoVLM will use.

The important idea is that **both modalities learn from each other**.

```text
Image ────────────────> Text
          Image-to-Text


Text ─────────────────> Image
          Text-to-Image

                ↓

          Symmetric Loss
```

---

# 13. Intuition Behind the Loss

The mathematics can initially look complicated, but the underlying idea is simple.

Imagine a batch containing:

```text
Image 1 → "red circle top-left"
Image 2 → "blue square center"
Image 3 → "green triangle bottom-right"
```

For Image 1, the model sees three possible captions:

```text
red circle top-left          ← correct
blue square center           ← incorrect
green triangle bottom-right  ← incorrect
```

The model produces similarities:

```text
0.91
0.20
0.10
```

That's good.

We want:

$$
S_{11} \gg S_{12}, S_{13}
$$

Now imagine the model produces:

```text
0.30
0.80
0.75
```

That's bad.

The correct caption has a lower similarity than the incorrect captions.

The contrastive loss penalizes this.

Therefore, training repeatedly pushes the model toward:

```text
              Matching pair
                   ↑
                   │
Image ─────────────┼──────────── Text
                   │
             high similarity
```

while pushing mismatched pairs apart:

```text
Image ───────── X ───────── Text
             low similarity
```

### The key intuition

The model is not directly learning a sentence such as:

> "A blue circle is at the top-left."

Instead, it learns a **geometry** where matching visual and textual concepts occupy nearby locations.

That is why contrastive learning is so powerful.

---

# 14. A Tiny Numerical Example

Let's make the entire process concrete.

Suppose we have a batch containing only two image-text pairs:

```text
I₁ ↔ T₁
I₂ ↔ T₂
```

Assume the embeddings are already normalized.

Let:

$$
z_{I_1} =
\begin{bmatrix}
1 \
0
\end{bmatrix},
\qquad
z_{I_2} =
\begin{bmatrix}
0 \
1
\end{bmatrix}
$$

and:

$$
z_{T_1} =
\begin{bmatrix}
1 \
0
\end{bmatrix},
\qquad
z_{T_2} =
\begin{bmatrix}
0.6 \
0.8
\end{bmatrix}
$$

---

## Step 1 — Calculate similarities

Using the dot product:

$$
S_{ij} = z_{I_i}^T z_{T_j}
$$

We obtain:

$$
S =
\begin{bmatrix}
1.0 & 0.0 \
0.6 & 0.8
\end{bmatrix}
$$

The diagonal represents the positive pairs:

$$
S_{11}=1.0,
\qquad
S_{22}=0.8
$$

---

## Step 2 — Apply temperature

Suppose:

$$
\tau = 0.5
$$

Then:

$$
\frac{S}{\tau}
==============

\begin{bmatrix}
2.0 & 0.0 \
1.2 & 1.6
\end{bmatrix}
$$

---

## Step 3 — Image-to-text probabilities

For Image 1:

$$
p(T_1 \mid I_1)
===============

\frac{e^2}{e^2+e^0}
\approx 0.881
$$

So the model assigns approximately:

```text
88.1% → correct caption
11.9% → incorrect caption
```

For Image 2:

$$
p(T_2 \mid I_2)
===============

\frac{e^{1.6}}{e^{1.2}+e^{1.6}}
\approx 0.599
$$

So:

```text
59.9% → correct caption
40.1% → incorrect caption
```

---

## Step 4 — Image-to-text loss

For Image 1:

$$
L_1 = -\log(0.881)
$$

For Image 2:

$$
L_2 = -\log(0.599)
$$

The average is approximately:

$$
L_{I\rightarrow T}
\approx 0.325
$$

---

## Step 5 — Text-to-image

We perform the same operation in the opposite direction.

For Text 1:

$$
p(I_1 \mid T_1)
===============

\frac{e^2}{e^2+e^{1.2}}
\approx 0.690
$$

For Text 2:

$$
p(I_2 \mid T_2)
===============

\frac{e^{1.6}}{e^{0}+e^{1.6}}
\approx 0.832
$$

Therefore:

$$
L_{T\rightarrow I}
\approx 0.321
$$

---

## Step 6 — Symmetric loss

Finally:

$$
L =
\frac{L_{I\rightarrow T}+L_{T\rightarrow I}}{2}
$$

Therefore:

$$
L =
\frac{0.325+0.321}{2}
$$

giving approximately:

$$
L \approx 0.323
$$

This is the number that would ultimately be backpropagated through our model.

---

# Putting Everything Together

Our entire NanoVLM training pipeline can now be summarized as:

```text
                         IMAGE
                           │
                           ▼
                    ┌─────────────┐
                    │  Image CNN  │
                    └──────┬──────┘
                           │
                           ▼
                    Image Embedding
                           │
                           │
                           │ Similarity
                           │ Matrix
                           ▼
                    Contrastive Loss
                           ▲
                           │
                           │
                    Text Embedding
                           ▲
                           │
                    ┌──────┴──────┐
                    │ Text Encoder│
                    └─────────────┘
                           ▲
                           │
                          TEXT
```

The training objective is:

**Make matching image-text embeddings similar**

and:

**Make mismatched image-text embeddings dissimilar**

This simple idea is the foundation on which we will build our NanoVLM.

---

## One important correction for our implementation

There's a subtle but important distinction we'll preserve in the code:

**Similarity matrix:**

$$
S = Z_I Z_T^T
$$

**Temperature-scaled logits:**

$$
\text{logits} = \frac{S}{\tau}
$$

**Loss:**

We should use PyTorch's numerically stable cross-entropy implementation rather than manually computing `exp()` and `log()` in the actual model.

Conceptually we have:

```python
loss_i2t = cross_entropy(logits, labels)
loss_t2i = cross_entropy(logits.T, labels)

loss = (loss_i2t + loss_t2i) / 2
```

This will make the eventual `loss.py` both clean and numerically stable.
